# Chapter 12. Graph Neural Networks for Chemistry
## Part 4. Diagnose representations and audit explanations

Good test predictions, distinguishable graph representations, and convincing explanations are separate requirements. This notebook isolates several failure modes using small controlled examples. It complements the measured-solubility workflow in [Part 3](Chapter12_Part3.ipynb), while requiring no saved checkpoint or training run from that notebook.

### Learning objectives

- Recognize oversmoothing in a specified averaging model without claiming that every deep GNN must behave that way.
- Construct two different graphs that a restricted message-passing architecture cannot distinguish.
- Separate an input gradient from a baseline-relative integrated-gradient attribution.
- Verify numerical completeness, baseline dependence, permutation consistency, and sensitivity to model parameters.
- Distinguish a tensor edit from a chemically valid counterfactual.
- Plan error, uncertainty, and explanation checks for a trained molecular model.

**Scope:** all models below are explicit numerical controls with fixed weights; no experimental chemical target is learned. The scores and attributions have arbitrary units. Everything runs offline on one CPU thread. Use a fresh kernel in the [course environment](Readme.md) and run the setup cell first.

### Start here: three questions that need different evidence

1. **Can the model see the difference?** Two inputs can become identical after encoding or aggregation. No training can recover information already discarded.
2. **Does the trained model predict well?** Answer using appropriate held-out measurements and baselines, as in Part 3.
3. **Why did this function give this score?** Integrated gradients divide a specified score difference among input entries. They describe the function being explained; they are not automatically evidence of a chemical mechanism.

Here a *gradient* means a local slope, a *baseline* means the reference input for a comparison, and *completeness* means that contributions sum to the specified score difference. Completeness is an accounting check, not a certificate of truth.

**First pass:** inspect the averaging animation-like snapshots, the two indistinguishable graphs, and the baseline-path plot. **Deeper pass:** matrix limits and integrated-gradient derivation. The research review exercise is to evaluate a claim that “the oxygen explains the activity”: first demand a defined target, held-out predictive evidence, and a declared reference. [Part 8](Chapter12_Part8.ipynb) applies related checks to a trained PyG classifier and an actual library explainer.

In [ ]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"

from pathlib import Path
from copy import deepcopy
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from IPython.display import display
import torch
from torch import nn
from rdkit import Chem, rdBase

OUT = Path("outputs/chapter12_part4")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 1204
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DTYPE = torch.float64
print(f"PyTorch {torch.__version__}; RDKit {rdBase.rdkitVersion}; CPU threads={torch.get_num_threads()}")

### 12.4.1. A controlled example of oversmoothing

Take an undirected connected path graph and update its node-feature matrix by repeated **lazy random-walk averaging**:

$$H^{(t+1)}=PH^{(t)},\qquad P=\frac12(I+D^{-1}A).$$

Here $A$ is the adjacency matrix and $D$ contains node degrees. A node keeps half its state and averages the other half over its neighbors. The self-retention makes this connected walk aperiodic. For this finite graph, repeated updates converge to identical node rows, each equal to the stationary-degree-weighted mean of the initial rows. That loss of node distinctions illustrates oversmoothing.

The conserved weights are $\pi_i=d_i/\sum_jd_j$, not generally $1/N$. We monitor the weighted disagreement $V_t=\sum_i\pi_i\|h_i^{(t)}-\bar h\|^2$, where $\bar h=\pi^\mathsf T H^{(0)}$. This fixed linear process has no trainable weights, activations, or task loss. It is a diagnostic example, not a theorem that every deep trained GNN or every residual network must collapse. Related analysis: [Li et al.](https://arxiv.org/abs/1801.07606).

In [ ]:
number_of_nodes = 8
A_path = np.zeros((number_of_nodes, number_of_nodes))
for i in range(number_of_nodes-1):
    A_path[i, i+1] = A_path[i+1, i] = 1
degree = A_path.sum(axis=1)
P = 0.5*(np.eye(number_of_nodes)+A_path/degree[:, None])
stationary = degree/degree.sum()
np.testing.assert_allclose(P.sum(axis=1), 1)
np.testing.assert_allclose(stationary @ P, stationary)
initial_features = np.column_stack([np.linspace(-1, 1, number_of_nodes),
                                   np.arange(number_of_nodes) % 2])
stationary_mean = stationary @ initial_features
history = [initial_features.copy()]
for _ in range(80):
    history.append(P @ history[-1])
history = np.stack(history)
disagreement = np.sum(stationary[None, :, None]*(history-stationary_mean)**2, axis=(1, 2))
np.testing.assert_allclose(np.einsum("n,tnf->tf", stationary, history),
                           np.tile(stationary_mean, (len(history), 1)), atol=1e-12)
assert disagreement[-1] < 0.005*disagreement[0]
print("Stationary weighted mean:", stationary_mean)
print(f"Weighted disagreement: {disagreement[0]:.6f} -> {disagreement[-1]:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout="constrained")
for depth in [0, 5, 20, 80]:
    axes[0].plot(range(number_of_nodes), history[depth, :, 0], "o-", label=f"{depth} rounds", markersize=4)
axes[0].axhline(stationary_mean[0], color="0.5", linestyle="--", label="Limiting weighted mean")
axes[0].set(xlabel="Node along the path", ylabel="First feature", title="Repeated linear neighbor averaging")
axes[0].legend(fontsize=8)
axes[1].semilogy(range(len(history)), disagreement, color="#2b7893")
axes[1].set(xlabel="Averaging rounds", ylabel="Stationary-weighted disagreement", title="Node distinctions decay in this control")
for ax in axes:
    ax.grid(alpha=0.2)
fig.savefig(OUT / "linear_oversmoothing.png", dpi=140, bbox_inches="tight")
plt.show()

**Depth has several different limitations.** Too few local rounds cannot carry a changed input across a long graph distance. Oversmoothing concerns loss of distinctions between node states. **Oversquashing** concerns many potentially relevant signals compressed into limited vectors, especially across graph bottlenecks; states need not become equal for it to occur. Residuals, normalization, altered connectivity, or global communication change the architecture, but their effect must be checked for the intended task. See [Alon & Yahav](https://arxiv.org/abs/2006.05205).

### 12.4.2. Different graphs can have identical local representations

Consider a six-node cycle and two disjoint three-node cycles. These are **abstract untyped graphs**, not molecular structures. All six nodes start with the same feature vector; every node has two neighbors. A shared update using only the current node state and a sum of neighboring states therefore produces the same new state at every node of both graphs, at every round. A sum readout also matches because both graphs have six nodes.

This is a simple 1-WL/message-passing limitation **under those input and architecture assumptions**. Adding a component-count feature, other structural features, or a more expressive architecture could distinguish the pair. The example does not claim that all GNNs or all chemical encodings collide on these graphs. See the [GIN/1-WL expressivity analysis](https://arxiv.org/abs/1810.00826).

In [ ]:
def adjacency_from_cycles(cycles, n=6):
    adjacency = np.zeros((n, n))
    for cycle in cycles:
        for i, j in zip(cycle, cycle[1:]+cycle[:1]):
            adjacency[i, j] = adjacency[j, i] = 1
    return adjacency

A_cycle = adjacency_from_cycles([[0, 1, 2, 3, 4, 5]])
A_triangles = adjacency_from_cycles([[0, 1, 2], [3, 4, 5]])
assert not np.array_equal(A_cycle, A_triangles)
np.testing.assert_array_equal(A_cycle.sum(1), np.full(6, 2))
np.testing.assert_array_equal(A_triangles.sum(1), np.full(6, 2))
W_self = np.array([[0.4, -0.2], [0.1, 0.3]])
W_neighbor = np.array([[0.2, 0.1], [-0.3, 0.5]])
left_states = right_states = np.ones((6, 2))
collision_rows = []
for depth in range(7):
    np.testing.assert_allclose(left_states, right_states, atol=1e-14)
    collision_rows.append({"round": depth, "max node difference": float(np.abs(left_states-right_states).max()),
                           "pooled difference": float(np.linalg.norm(left_states.sum(0)-right_states.sum(0)))})
    left_states = np.tanh(left_states @ W_self + (A_cycle @ left_states) @ W_neighbor)
    right_states = np.tanh(right_states @ W_self + (A_triangles @ right_states) @ W_neighbor)
display(pd.DataFrame(collision_rows))
triangle_counts = [int(round(np.trace(A @ A @ A)/6)) for A in [A_cycle, A_triangles]]
assert triangle_counts == [0, 2]
print("Triangles in the two abstract graphs:", triangle_counts)

In [ ]:
angles = np.linspace(0, 2*np.pi, 6, endpoint=False)
cycle_xy = np.column_stack([np.cos(angles), np.sin(angles)])
triangle_angles = np.linspace(0, 2*np.pi, 3, endpoint=False)
triangle = 0.65*np.column_stack([np.cos(triangle_angles), np.sin(triangle_angles)])
triangles_xy = np.vstack([triangle+[-1, 0], triangle+[1, 0]])
fig, axes = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")
for ax, adjacency, xy, title in zip(axes, [A_cycle, A_triangles], [cycle_xy, triangles_xy],
                                    ["Six-node cycle", "Two disjoint three-node cycles"]):
    for i, j in zip(*np.where(np.triu(adjacency, 1))):
        ax.plot(xy[[i, j], 0], xy[[i, j], 1], color="0.5", zorder=1)
    ax.scatter(xy[:, 0], xy[:, 1], s=320, color="#dbe8f0", edgecolors="#2b7893", zorder=2)
    for i, point in enumerate(xy):
        ax.text(*point, str(i), ha="center", va="center", zorder=3)
    ax.set(title=title, aspect="equal")
    ax.margins(0.2)
    ax.axis("off")
fig.suptitle("Abstract graphs: identical initial node features, degree 2 everywhere")
fig.savefig(OUT / "graph_collision.png", dpi=140, bbox_inches="tight")
plt.show()

### 12.4.3. Define the score before explaining it

Now use ethanol's heavy-atom connectivity and two input entries per atom: carbon and oxygen indicators. This encoding omits hydrogen counts, charge, stereochemistry, and experimental conditions. The following one-round function has **fixed declared weights**:

$$H=\tanh(XW_s^\mathsf T+AXW_n^\mathsf T+b_h),\qquad
F(X;A)=\left(\sum_i h_i\right)\cdot w_r+b_r.$$

The adjacency $A$ is fixed when differentiating input features. All matrices are printed and assigned explicitly. No weights are fitted, so $F$ is an arbitrary smooth graph score with no chemical target or physical units. This controlled setting lets us test attribution arithmetic without presenting an attractive heatmap as a learned chemical mechanism.

In [ ]:
ethanol = Chem.MolFromSmiles("CCO")
X = torch.tensor([[float(atom.GetAtomicNum() == 6), float(atom.GetAtomicNum() == 8)]
                  for atom in ethanol.GetAtoms()], dtype=DTYPE)
A = torch.tensor(Chem.GetAdjacencyMatrix(ethanol), dtype=DTYPE)
assert X.shape == (3, 2) and A.shape == (3, 3)

class DeclaredGraphScore(nn.Module):
    def __init__(self):
        super().__init__()
        self.self_weight = nn.Parameter(torch.tensor([[0.6, 0.1], [-0.2, 1.1]], dtype=DTYPE))
        self.neighbor_weight = nn.Parameter(torch.tensor([[0.2, 0.7], [0.5, -0.1]], dtype=DTYPE))
        self.hidden_bias = nn.Parameter(torch.tensor([0.1, -0.1], dtype=DTYPE))
        self.readout_weight = nn.Parameter(torch.tensor([0.8, -0.6], dtype=DTYPE))
        self.readout_bias = nn.Parameter(torch.tensor(0.15, dtype=DTYPE))

    def forward(self, features, adjacency):
        # Leading dimensions allow a batch of interpolation points on the same graph.
        neighbor_features = adjacency @ features
        hidden = torch.tanh(features @ self.self_weight.T + neighbor_features @ self.neighbor_weight.T
                            + self.hidden_bias)
        return hidden.sum(dim=-2) @ self.readout_weight + self.readout_bias

score_model = DeclaredGraphScore().eval()
for name, parameter in score_model.named_parameters():
    print(name, parameter.detach().tolist())
with torch.inference_mode():
    original_score = score_model(X, A).item()
print(f"Declared score on ethanol's encoded graph: {original_score:.6f} (arbitrary units)")

### Input gradients describe a local continuous sensitivity

The derivative $\partial F/\partial X_{if}$ measures the score's infinitesimal response to that numerical feature, while all other entries and the adjacency stay fixed. It is not the effect of an actual atom replacement. Moving a one-hot indicator continuously produces values that need not represent any molecule; the chosen feature coordinates and scaling also affect raw gradient values. Saturated activations can yield small local gradients even when the score differs substantially from a chosen reference.

We check one derivative with finite differences. A correct derivative is necessary for the later calculation; it does not validate the score as chemistry.

In [ ]:
gradient_input = X.clone().requires_grad_(True)
input_gradient = torch.autograd.grad(score_model(gradient_input, A), gradient_input)[0]
step = 1e-5
plus_x, minus_x = X.clone(), X.clone()
plus_x[2, 1] += step
minus_x[2, 1] -= step
with torch.inference_mode():
    finite_difference = (score_model(plus_x, A)-score_model(minus_x, A))/(2*step)
torch.testing.assert_close(input_gradient[2, 1], finite_difference, atol=1e-8, rtol=1e-6)
display(pd.DataFrame(input_gradient.numpy(), columns=["dF/d carbon entry", "dF/d oxygen entry"])
        .rename_axis("atom_index"))
print("Oxygen-entry gradient and finite difference:", input_gradient[2, 1].item(), finite_difference.item())

### 12.4.4. Integrated gradients explain a specified contrast

For input $X$ and baseline $X_0$, integrated gradients (IG) along a straight path are

$$\mathrm{IG}_{if}(X;X_0)=(X_{if}-X_{0,if})
\int_0^1\frac{\partial F(X_0+\alpha(X-X_0);A)}{\partial X_{if}}\,d\alpha.$$

For this differentiable function, summing all entries gives **completeness**:

$$\sum_{i,f}\mathrm{IG}_{if}=F(X;A)-F(X_0;A).$$

This is a decomposition of a model-score difference. It does not state that individual attributions are unique physical causes. Numerical quadrature introduces integration error, which we report and check using [PyTorch's trapezoidal integration](https://docs.pytorch.org/docs/2.11/generated/torch.trapezoid.html). Source: [Sundararajan et al., Integrated Gradients](https://proceedings.mlr.press/v70/sundararajan17a.html).

We compare two deliberately nonchemical baselines on the same topology: all-zero features, and the same vector $[2/3,1/3]$ at every atom. The latter is chosen as an illustrative contrast, not a statistic learned from an experimental training set. Both the zero baseline and intermediate fractional element indicators can lie outside the set of valid molecular encodings. Changing the baseline changes the question being explained.

In [ ]:
def integrated_gradients(model, features, adjacency, baseline, points=257):
    if features.shape != baseline.shape or points < 2:
        raise ValueError("Use a matching baseline and at least two integration points.")
    alpha = torch.linspace(0, 1, points, dtype=features.dtype)
    path = (baseline.unsqueeze(0)+alpha[:, None, None]*(features-baseline).unsqueeze(0)).requires_grad_(True)
    path_scores = model(path, adjacency)
    assert path_scores.shape == (points,)
    path_gradients = torch.autograd.grad(path_scores.sum(), path)[0]
    average_gradient = torch.trapezoid(path_gradients, alpha, dim=0)
    attribution = (features-baseline)*average_gradient
    with torch.inference_mode():
        contrast = (model(features, adjacency)-model(baseline, adjacency)).item()
    residual = attribution.sum().item()-contrast
    return attribution.detach(), contrast, residual

BASELINES = {"zero features": torch.zeros_like(X),
             "uniform fractional features": torch.tensor([[2/3, 1/3]], dtype=DTYPE).repeat(3, 1)}
attributions = {}
completeness_rows = []
for label, baseline in BASELINES.items():
    attribution, contrast, residual = integrated_gradients(score_model, X, A, baseline)
    assert abs(residual) < 2e-5
    attributions[label] = attribution
    completeness_rows.append({"baseline": label, "F(input)-F(baseline)": contrast,
                              "sum of IG": attribution.sum().item(), "completeness residual": residual})
completeness = pd.DataFrame(completeness_rows)
display(completeness)
coarse_ig, _, coarse_residual = integrated_gradients(score_model, X, A, BASELINES["zero features"], points=65)
print(f"Zero-baseline integration residual, 65 points: {coarse_residual:.2e}; 257 points: {completeness_rows[0]['completeness residual']:.2e}")
print("IG is summed across features to obtain each atom's displayed contribution.")

### Visual bridge: the baseline defines the question being explained

Follow each path from reference features at $\alpha=0$ to the same input at $\alpha=1$. Its vertical change is the score difference that integrated gradients must distribute. The paths share an endpoint but start from different scores, so their total attributions need not agree. We compute the curves from the declared function above; the lines are not fitted experimental trends.

Both reference inputs and intermediate fractional element indicators are computational constructs. They are not molecules that could be synthesized. When reviewing a research explanation, ask “relative to which input?” before assigning chemical meaning to the colored atoms. A useful follow-up is a physically valid matched molecular pair, measured under comparable conditions; this notebook does not supply such causal evidence.

In [ ]:
interpolation_fraction = torch.linspace(0,1,41,dtype=DTYPE)
fig, axes = plt.subplots(1,2,figsize=(10.5,3.5),layout='constrained')
for label,baseline in BASELINES.items():
    path = baseline[None] + interpolation_fraction[:,None,None]*(X-baseline)[None]
    with torch.inference_mode():
        curve = score_model(path,A).numpy()
    axes[0].plot(interpolation_fraction.numpy(),curve,label=label)
    axes[1].plot(interpolation_fraction.numpy(),curve-curve[0],label=label)
axes[0].set(xlabel='Path fraction α',ylabel='Declared score (arbitrary units)',title='Same endpoint, different references')
axes[1].set(xlabel='Path fraction α',ylabel='Score minus reference score',title='Different contrasts to distribute')
axes[0].legend(fontsize=8)
for ax in axes: ax.grid(alpha=.2)
fig.savefig(OUT / 'attribution_reference_paths.png',dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout="constrained")
zero_ig = attributions["zero features"].numpy()
limit = float(np.abs(zero_ig).max())
im = axes[0].imshow(zero_ig, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
axes[0].set_xticks([0, 1], ["Carbon entry", "Oxygen entry"])
axes[0].set_yticks([0, 1, 2], ["0: C", "1: C", "2: O"])
axes[0].set(title="Feature IG from the zero baseline", ylabel="Atom index")
for i in range(3):
    for j in range(2):
        label = f"{zero_ig[i, j]:+.3f}" if zero_ig[i, j] != 0 else "0"
        axes[0].text(j, i, label, ha="center", va="center",
                     color="white" if abs(zero_ig[i, j]) > 0.65*limit else "black")
fig.colorbar(im, ax=axes[0], label="Score attribution (arbitrary units)")
positions = np.arange(3)
for offset, (label, attribution) in zip([-0.18, 0.18], attributions.items()):
    axes[1].bar(positions+offset, attribution.sum(dim=1).numpy(), width=0.35, label=label)
axes[1].axhline(0, color="0.4", linewidth=0.8)
axes[1].set_xticks(positions, ["0: C", "1: C", "2: O"])
axes[1].set(xlabel="Atom index", ylabel="Sum of feature IG", title="Different baselines explain different contrasts")
axes[1].legend(fontsize=7.5)
axes[1].grid(axis="y", alpha=0.2)
fig.savefig(OUT / "baseline_attributions.png", dpi=140, bbox_inches="tight")
plt.show()

Positive and negative contributions refer to the score difference for the chosen baseline. A zero contribution can arise because an entry is unchanged along the path, even if its local derivative is nonzero. Summing an atom's feature attributions can also hide cancellation between features. Read the feature-level values and the baseline with the atom-level plot.

### 12.4.5. Check that an attribution responds to the model

Attributions should respect a consistent relabeling of atoms. They should also be investigated when model parameters change: an explanation method that ignores the model may still draw visually plausible pictures. We change all declared parameters and show that IG changes in this example, while checking completeness again. This is a small parameter-sensitivity check inspired by [Adebayo et al.](https://arxiv.org/abs/1810.03292); it is not their full experimental protocol and does not by itself validate an explanation method.

For a trained model, useful further checks include progressively randomizing layers, comparing several valid contrasts, testing repeated runs, and auditing held-out errors. Label-randomization tests require fitting an appropriate control model; there is no trained-label claim to test in this fixed-weight demonstration.

In [ ]:
permutation = torch.tensor([2, 0, 1])
permuted_x = X[permutation]
permuted_a = A[permutation][:, permutation]
permuted_baseline = BASELINES["zero features"][permutation]
permuted_ig, _, _ = integrated_gradients(score_model, permuted_x, permuted_a, permuted_baseline)
torch.testing.assert_close(permuted_ig, attributions["zero features"][permutation], atol=1e-10, rtol=1e-10)

randomized_model = deepcopy(score_model)
generator = torch.Generator().manual_seed(SEED+1)
with torch.no_grad():
    for parameter in randomized_model.parameters():
        parameter.copy_(0.4*torch.randn(parameter.shape, generator=generator, dtype=DTYPE))
randomized_ig, randomized_contrast, randomized_residual = integrated_gradients(
    randomized_model, X, A, BASELINES["zero features"])
assert abs(randomized_residual) < 2e-5
attribution_change = (randomized_ig-attributions["zero features"]).abs().sum().item()
assert attribution_change > 1e-3
display(pd.DataFrame({"atom": ["0: C", "1: C", "2: O"],
                      "declared weights IG": attributions["zero features"].sum(1).numpy(),
                      "randomized weights IG": randomized_ig.sum(1).numpy()}))
print(f"Attribution L1 change after parameter randomization: {attribution_change:.6f}")
print(f"Randomized-model completeness residual: {randomized_residual:.2e}")
print("Atom-permutation consistency passed.")

### 12.4.6. Tensor removal is not automatically a molecular counterfactual

Removing a bond from the adjacency or zeroing an atom-feature row changes the computational input. It does not necessarily produce a valid molecule, preserve the charge/protonation state, or stay within a model's training domain. A chemically meaningful edit must define a new structure, validate it, recompute its features, and specify the property conditions. Even then, a predicted difference is a model prediction to be evaluated.

The controls below deliberately **keep the other features fixed**. We remove both directions of the C–O edge, or zero the oxygen feature row while keeping its edges. They answer different tensor-perturbation questions and need not equal oxygen's IG contribution. We do not interpret these controls as bond dissociation energies, measured solubility changes, or chemical mechanisms.

In [ ]:
deleted_bond = A.clone()
deleted_bond[1, 2] = deleted_bond[2, 1] = 0
zeroed_oxygen = X.clone()
zeroed_oxygen[2] = 0
with torch.inference_mode():
    tensor_controls = pd.DataFrame([
        {"control": "unchanged encoding", "score": score_model(X, A).item()},
        {"control": "remove C-O adjacency entries; other features fixed", "score": score_model(X, deleted_bond).item()},
        {"control": "zero oxygen row; adjacency fixed", "score": score_model(zeroed_oxygen, A).item()}])
tensor_controls["change from original"] = tensor_controls["score"]-original_score
display(tensor_controls)
print("These are computational controls, not validated chemical edits.")

### 12.4.7. Transfer these checks to a trained molecular model

For the model in Part 3, begin with the prediction task and its held-out errors before interpreting hidden vectors or saliency maps.

| Question | Concrete check | What a pass does not prove |
|---|---|---|
| Is evaluation independent? | Keep related molecules/representations in the intended split; fit preprocessing and model selection without test labels. | Performance on every future scaffold, assay, or laboratory. |
| Does the model beat a useful alternative? | Compare training-mean, descriptor/fingerprint, and GNN baselines under the same split and metric. | A causal reason for the improvement. |
| Which cases fail? | Inspect residuals by target range, scaffold, charge state, size, and similarity to training molecules; record sample counts. | Reliable subgroup estimates when groups are tiny. |
| Are numerical predictions stable? | Repeat with prespecified seeds or training resamples, while retaining the same test protocol. | That shared model bias has disappeared. |
| Is the explanation tied to the stated score? | Check units, baseline, completeness when applicable, permutation consistency, and parameter sensitivity. | A chemical mechanism or a physically valid intervention. |
| Is the representation adequate? | Audit stereochemistry, ionic state, geometry, and assay context for the endpoint. | That increasing network depth will supply omitted information. |

**Similarity is not an uncertainty estimate by itself.** A nearest-training fingerprint similarity is a representation-dependent description of neighborhood. It can help stratify errors, but high similarity does not guarantee accuracy and low similarity does not imply a particular error bar.

**Ensemble spread is not a guaranteed confidence interval.** Independently fitted models can disagree because of training variability; they can also agree while sharing a systematic error. [Deep ensembles](https://arxiv.org/abs/1612.01474) provide one practical uncertainty approach, whose behavior still needs evaluation on data representative of the use case. A deterministic score alone does not supply predictive uncertainty.

**Calibration is a separate empirical question.** In binary classification, among cases assigned positive-class probability 0.8, about 80% should have the positive label within a suitable population; accuracy alone does not ensure this. In regression, proposed prediction intervals need an explicit construction and coverage/width evaluation. Fit calibration choices without the final test labels, and evaluate distribution shifts separately. [Guo et al.](https://proceedings.mlr.press/v70/guo17a.html) examine classification calibration, not a universal regression-interval guarantee.

Finally, learned attention coefficients are part of a model's computation. They can be useful to inspect, but they are not automatically a faithful attribution or a causal explanation; that requires a specified claim and appropriate checks. See the primary study [Jain & Wallace](https://aclanthology.org/N19-1357/) and the discussion [Wiegreffe & Pinter](https://aclanthology.org/D19-1002/). Neither an appealing picture nor an architecture name replaces validation.

In [ ]:
completeness.to_csv(OUT / "integrated_gradient_completeness.csv", index=False)
tensor_controls.to_csv(OUT / "tensor_controls.csv", index=False)
record = {"scope": "Controlled linear mixing, abstract graph collision, and fixed-weight score attribution",
          "experimental_target": None, "training_performed": False, "seed": SEED,
          "torch_version": str(torch.__version__), "rdkit_version": rdBase.rdkitVersion,
          "fixed_parameters": {name: parameter.detach().tolist() for name, parameter in score_model.named_parameters()},
          "smiles_for_connectivity": "CCO", "feature_order": ["carbon indicator", "oxygen indicator"],
          "original_score": original_score,
          "baselines": {name: baseline.tolist() for name, baseline in BASELINES.items()},
          "IG_zero_baseline": attributions["zero features"].tolist(),
          "integration_points": 257, "parameter_randomization_L1_change": attribution_change,
          "checks": ["stationary mean conserved", "linear disagreement decays", "regular graph collision",
                     "input gradient finite difference", "IG completeness", "permutation consistency",
                     "parameter sensitivity"]}
(OUT / "diagnostics.json").write_text(json.dumps(record, indent=2)+"\n", encoding="utf-8")
print("Saved diagnostics, exact fixed parameters, attribution settings, and figures under", OUT)

### Exercises

1. Why is the limiting average degree-weighted for the specified lazy random walk? Would an ordinary arithmetic mean always give the same answer?
2. Distinguish limited receptive field, oversmoothing, and oversquashing. Which did the first experiment demonstrate directly?
3. Prove by induction that both regular abstract graphs keep identical node states under the stated shared sum-update rule. Name an extra graph feature that would distinguish them.
4. What is held fixed in the input derivative? Why is a small gradient not necessarily evidence that an atom is chemically irrelevant?
5. State the completeness equation and its numerical residual. Does completeness require a zero baseline?
6. Explain why an entry with nonzero local gradient can have zero IG when that feature is unchanged from the baseline. Why can changing the baseline change atom contributions?
7. What did parameter randomization check here? Why is a passing result insufficient to establish a chemical mechanism?
8. Why must both adjacency directions be removed in the bond-removal control? What extra work would a valid molecular edit require?
9. Contrast nearest-training similarity, ensemble spread, and calibrated predictive uncertainty.

<details><summary>Suggested answers</summary>

1. For this undirected row-normalized walk, $\pi_i\propto d_i$ satisfies $\pi^\mathsf TP=\pi^\mathsf T$. Uniform weighting generally differs when degrees vary.
2. A signal has not traveled far enough; node representations lose distinctions; or many signals are compressed through limited channels. The first experiment directly demonstrates oversmoothing of one fixed linear process.
3. Initially all states agree. Every node receives two copies of the same state, so a shared update preserves equality; six-node pooling also agrees. Component count or triangle count separates these graphs if provided.
4. Other numerical features and the adjacency. The derivative is coordinate-dependent and local, may be affected by saturation, and does not implement a chemical change.
5. Sum of all IG entries equals $F(X;A)-F(X_0;A)$ for the differentiable path, with quadrature error in the code. Any specified matching baseline can be used; its scientific interpretation is a separate issue.
6. IG multiplies the path-averaged derivative by the feature difference. A changed baseline changes both that difference and the integration path, and often changes the total score contrast.
7. IG responded to changed model parameters in this controlled example while retaining completeness. The model has no measured target, and its inputs/paths are not chemical interventions.
8. The underlying bond is undirected; deleting only one edge changes message direction asymmetrically. A chemical edit needs a valid new structure, sanitization/state decisions, recomputed features, and a defined property comparison.
9. Similarity describes proximity under an encoding; ensemble spread describes disagreement among fitted models; calibrated uncertainty needs empirical agreement with stated probability or interval behavior on the relevant population. None automatically guarantees reliability under distribution shift.

</details>

[Previous: measured-data learning](Chapter12_Part3.ipynb) · [Next: 3D geometry and equivariance](Chapter12_Part5.ipynb) · [Course contents](Readme.md)